# SULO  Clinical Data Modeling Tutorial

This tutorial use the SULO ontology for representing typical healthcare concepts such as **Condition**, which refers to a person's health state as a *process* that the patient is experiencing; a **MedicalProcedure**, as an *activity or process* to assess, diagnose, treat or prevent a health condition; and **Measurement**, which is an *information entity* that represents the result of observing and/or quantifying some quality of a person, object or process. Measurements are always connected to sume qualitative and quantitative value (+ unit).

Along this tutorial we will represent an administrative case which describes a patient's encounter or stay at a healthcare provider institution. It can contain all clinical processes associated with the patient stay. E.g the admission and discharge events, observations, diagnoses, treatments or any assessment made to a patient. It may also include additional information such as the subject of care, or the location where the encounter or stay takes place.

Our administrative case will represent a care episode that includes the patient's admission to, and subsequent discharge from the healthcare facility. During the encounter/visit, the patient's blood pressure is measured, and hypertension is diagnosed. The case will also document the dministration of medication and the issuance of a prescription for continued treatment following discharge.

First, we will initialize the environment as follows:

In [ ]:
!pip install owlready2 # Install dependencies
import os

def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
install_java()

!rm -rf /content/*
!git clone https://github.com/CMCosta/SULO-health-data-modelling.git
%cd /content/SULO-health-data-modelling

!ls -lah lib
import sys
sys.path.insert(0, os.getcwd())

 Now, we will create our new ontology, which will import SULO. We will load the SULO ontology from it's web location and import it into our health ontology, so we can use SULO's classes and properties in our definitions.

In [ ]:
from owlready2 import *
from lib.helpers import *


# Load SULO ontology (assuming it's accessible at this URI)
sulo = get_ontology("https://w3id.org/sulo/sulo.owl").load()

# Define the new Healthcare-related extension ontology
health_ontology = get_ontology("https://w3id.org/sulo/extension/healthcare/")
health_ontology.imported_ontologies.append(sulo)

## 1. CREATION OF SOME OF THE MAIN CLINICAL ENTITY TYPES (as OWL classes)

We will create some of the main ontology classes needed for representing the administrative case mentioned above. For that, we need to classify these classes under the appropriate SULO top-level categories.

We will create classes for representing:

- **AdministrativeCase**: A process involving administrative and clinical events that occur during a care episode.
- **Admission**: A process by which a patient is accepted into a healthcare facility.
- **Discharge**: A process by which a patient is released from a healthcare facility.
- **Clinical Visit**: A process by which a patient visit  healthcare facility for healthcare service(s) or assessing the health status.
- **Condition**: A process of health, disorder, or disease.
- **Measurement**: A quantitative or qualitative result of some measurement procedure.
- **MeasurementProcess**: A process used to determine the value of a specific physical, physiological, or clinical parameter in a patient.
- **MedicationAdministration**: A process for representing the administration of medication.
- **MedicationPrescription**: A plan for administering medication, including the administration instructions.
- **PharmaceuticalProduct**: A clinical drug or medicinal product.
- **Patient**: A human who is the subject of care.
- **BodySite**: A distinct anatomical structure or region of a living organism’s body.
- **SubjectOfCareRole**: A role in which an individual is the focus of healthcare activities.
- **CareProviderRole**: A Role in which an individual has to provide a healthcare service.

Next, we classify them under the right SULO top-level class:

In [ ]:
with health_ontology:
    class AdministrativeCase (sulo.Process):
        pass
    class Admission (sulo.Process):
        pass
    class Discharge (sulo.Process):
        pass
    class Discharge (sulo.Process):
        pass
    class Condition (sulo.Process):
        pass
    class Measurement (sulo.Quantity):
        pass
    class MeasurementProcess (sulo.Process):
        pass
    class MedicationAdministration (sulo.Process):
        pass
    class MedicationPrescription (sulo.InformationObject):
        pass
    class PharmaceuticalProduct(sulo.SpatialObject):
        pass
    class HumanOrganism (sulo.SpatialObject):
        pass
    class Person (HumanOrganism):
        pass
    class BodySite (sulo.SpatialObject):
        pass
    class SubjectOfCareRole (sulo.Role):
        pass
    class CareProviderRole (sulo.Role):
        pass
    class PerformerRole (sulo.Role):
        pass
    class CareUnit (sulo.SpatialObject):
        pass
    print("Main classes or entity types created")

### 2. PROCESSES, THEIR PARTICIPANTS, AND THE ROLES THEY PLAY - MEASUREMENT PROCESS

A process might involve many different objects, each playing a different *role*. To discriminate the participants we instantiate roles for each individual involved in the process. This is why SULO uses only the object property *hasParticipant* and no subproperties, according to its philosophy of keeping the set of object and datatype properties to a minimum.

In SULO we have defined the PRO (Process-Role-Object) Design pattern to provide a modular way to represent how (non-process) entities participate in processes through their specific roles. The pattern requires that particular roles are directly linked to a process instance using
hasParticipant and to their respective role holders using isFeatureOf. The pattern removes the need for
role-based relations such as hasInstrument, hasOutcome, hasPatient, etc.

In the running use case we want to represent a measurement process, the measurement of blood pressure. This is described as a **MeasurementProcess** that occurs or takes place at some **BodyPart**, has as result some **Measurement** and is measured using some **Device**.


As we can see below, both the quantity result of the measurement and the device used to measure, are both participants of the **MeasurementProcess**, with the roles **OutputRole** and **InstrumentRole** respectively.

In [ ]:
with health_ontology:
    class ProcessStatus(sulo.Quality):
        pass
    class OutputRole (sulo.Role):
        pass
    class InstrumentRole (sulo.Role):
        pass
    class Device (sulo.SpatialObject):
        pass
    class MeasurementProcess (sulo.Process):
        is_a = [
            sulo.hasParticipant.some(OutputRole & sulo.isFeatureOf.some(Measurement)),
            sulo.hasParticipant.some(BodySite),
            sulo.hasFeature.some(ProcessStatus),
            sulo.hasParticipant.some(InstrumentRole & sulo.isFeatureOf.some(Device)),

            sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime)),
            sulo.hasParticipant.some(PerformerRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
        ]
    print("Axioms added to MeasurementProcedure")

In [ ]:
with health_ontology:
    class MedicalProcedure(sulo.Process):
        is_a = [
            sulo.hasParticipant.some(PharmaceuticalProduct),
            sulo.hasParticipant.some(BodySite),
            sulo.hasFeature.some(ProcessStatus),
            sulo.hasParticipant.some(InstrumentRole & sulo.isFeatureOf.some(Device)),

            sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime)),
            sulo.hasParticipant.some(PerformerRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
        ]
    print("Medication administration and related classes created")

### 3. QUANTITATIVE PROCESS OUTPUTS - QUANTITATIVE OUTPUT OF A MEASUREMENT

Once we have represented the **MeasurementProcess** class, let's model its quantitative result with the class **Measurement**, which is an *information object*, more specifically a *quantity*. In our running use case we are measuring blood pressure, which is a *quality* (**sulo.Quality**) of a *spatial object*, the arterial blood,  and its value has some **sulo.Unit** as a reference. Here we use the SOLID (Single Object Literal Information Datum) Design Pattern, which uses the functional datatype property, *hasValue*, to assign a literal value to an instance of an *InformationObject*. This pattern transforms the introduction of domain-specific data properties such as *hasBloodPressure* by first extracting the implied classes (e.g., Blood pressure) that pertain to a relevant *InformationObject* (e.g. Measurement), and secondly finding the right relation to associate the information object to either a process or an object.

In [ ]:
with health_ontology:
   class Measurement (sulo.Quantity):
        is_a =[sulo.refersTo.some(sulo.Quality),
               sulo.hasValue.some(float),
               sulo.hasPart.some(sulo.Unit),
               sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime))
        ]
   print("Axioms added to Measurement")

Next, we create specific classes for representing the blood pressure (**BloodPressure**) and the systolic blood pressure  (**SystolicBloodPressure**) and diastolic blood pressure  (**DiastolicBloodPressure**) results.

In [ ]:
with health_ontology:

    # --- Reuse or define minimal classes ---
    class BloodPressure(sulo.Quality): pass
    class SystolicBloodPressure(BloodPressure): pass
    class DiastolicBloodPressure(BloodPressure): pass

### 4. PROCESSES OCCUR IN SOME PLACE AND AT SOME TIME

We may want to register the (measured) time in which some process happened, whether in time instants, time intervals, or durations of time. In our running example this is for instance the case of the **Admission** and **Discharge** events. For both of them we have the instant of time (measurement) in which they occured. We will use the SULO property *atTime*. Note that all instances of time in SULO are results of time measurements, because the "absolute" time can only be approximated. In addition, we will represent where both processes happened (e.g. some healthcare facility) using the SULO property *isIn*.
Let's see how we can represent this using SULO.

In [ ]:
with health_ontology:
    class Admission (sulo.Process):
        is_a = [
            sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime)),
            sulo.isIn.some(CareUnit),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(CareProviderRole & sulo.isFeatureOf.some(Person))
        ]
    class Discharge (sulo.Process):
        is_a = [
            sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime)),
            sulo.isIn.some(CareUnit),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(CareProviderRole & sulo.isFeatureOf.some(Person))
        ]
    class ClinicalVisit (sulo.Process):
        is_a = [
            sulo.atTime.some(sulo.TimeInstant & sulo.hasValue.some(datetime.datetime)),
            sulo.isIn.some(CareUnit),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(CareProviderRole & sulo.isFeatureOf.some(Person))
        ]
    print("Temporal and location axioms added to Clinical Visit, Admission and Discharge")

### 5. HEALTHCARE CONDITIONS - DIAGNOSIS STATEMENT, CONDITION DIAGNOSED AND DIAGNOSTIC PROCESS

In our running example we want to represent an hypertension diagnosis. It is important to distinguish between the condition being diagnosed (**Condition**), which is a *process* in SULO, the diagnosis statement (**DiagnosisStatement**) wich is an *information object* and the diagnostic process (**DiagnosticProcess**) which is a *process*. In SULO we relate the condition (**Condition**) and the diagnosis statement (**DiagnosisStatement**) by using the relation *refersTo*, and this is a participant of the diagnostic process with the output role, since the statement is the result of the process.

In [ ]:
with health_ontology:
    class DiagnosisStatement(sulo.InformationObject):
        is_a = [
            sulo.refersTo.some(Condition)
        ]
    class DiagnosticProcess(sulo.Process):
        is_a = [
            sulo.hasParticipant.some(OutputRole & sulo.isFeatureOf.some(DiagnosisStatement))
        ]
    print("Diagnosis related classes created")

### 6. PROCESSES IN DIFFERENT STAGES, FROM PLANNING TO COMPLETION. MEDICATION ADMINISTRATION AND PRESCRIPTION

Clinical data refer to processes in different stages, from planning to completion. We will represent an already completed medication administration process as well as the prescription of some medication, in which the administration has not started yet.

To represent the administration of some medication, we first will create the **MedicationAdministration** class , which is a *process* in SULO. As described previously, in the process and their participants section, the specific medication or clinical drug (**PharmaceuticalProduct**) is one of the participants of the administration process. For the medication, we might need to indicate its dose (**PharmaceuticalDose**), which is a *quantity* in SULO as well as its dose form (**PharmaceuticalDoseForm**), which is a *quality* of the product.

In addition, we indicate the status of the administration process, **Completed**, as a subclass of **ProcessStatus**, which is a *quality* of the administration process in SULO. We can also indicate the time instant in which the medication was administered by using the SULO property *atTime*. The route of the administration by using *isIn* and indicating the body part (**BodyPart**) and the device **Device** used for administering it, if any. Last but not least, we can specify the patient and the person who administered the medication. All of them are participants of the administration process with their respective roles.

In [ ]:
with health_ontology:
    class Substance(sulo.SpatialObject):
        pass
    class SubstanceStrength(sulo.Quality):
        pass
    class PharmaceuticalDose(sulo.Quantity):
        sulo.refersTo.some(SubstanceStrength)
        sulo.hasValue.some(float),
        sulo.hasPart.some(sulo.Unit)
    class PharmaceuticalDoseForm(sulo.Quality):
        pass
    class PharmaceuticalProduct(sulo.SpatialObject):
        is_a = [
            x,
            sulo.hasFeature.some(PharmaceuticalDoseForm),
            sulo.hasDirectPart.some(Substance)
        ]
    class ProcessStatus(sulo.Quality):
        pass
    class Completed(ProcessStatus):
        pass
    class MedicationAdministration(sulo.Process):
        is_a = [
            sulo.hasParticipant.some(PharmaceuticalProduct),
            sulo.hasFeature.some(ProcessStatus),
            sulo.atTime.some(sulo.StartTime & sulo.hasValue.some(datetime.datetime)),
            sulo.atTime.some(sulo.EndTime & sulo.hasValue.some(datetime.datetime)),
            sulo.hasParticipant.some(InstrumentRole & sulo.isFeatureOf.some(Device)),
            sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(Person)),
            sulo.hasParticipant.some(BodySite)
        ]
    print("Medication administration and related classes created")

The above representation indicates that the administration of certain medication was performed. However, we might want to describe that there is a prescription for such administration **MedicationPrescription** which is a treatment plan and thus an *information object* in SULO, but the administration has not been perfomed yet, which we will indicate as a feature or quality of the administration process which indicate its status as **NotStarted**. In the following we create the class for the **MedicationPrescription** and indicate that it refers to some **MedicationAdministration** process. Within the **MedicationAdministration** class, in this case the dose, dose form, etc. will indicate the instructions to actually administer the clinical drug. The reason of referreing to a collection (plurality) is that an individual administration process does not exist in this stage of prescription, and perhaps not even in the future (the patient refuses to buy or to ingest the drug). A direct reference from an OWL class to another OWL class is not available in OWL-DL (it would require so-called punning). As an approximation SULO recommends the reference to a collection, which can be seen as the set of the instances of an OWL class. A complete ontological analysis of that is still under discussion).     

In [ ]:
with health_ontology:
    class MedicationPrescription(sulo.InformationObject):
        sulo.refersTo.some(sulo.Collection & sulo.hasItem.some(MedicationAdministration))


### 7. PROCESSES AND PARTS

All the processes described above are part of our administrative case. In SULO, a process can have another processes as parts. We will indicate that **Admission**, **Discharge**, the **MeasurementProcess** and the **DiagnosticProcess** are all parts of the process **AdministrativeCase**. We will also state that it can only have one **Admission** and one **Discharge** processes maximum. Finally we will add the Patient as the subject of care to the case.

In [ ]:
with health_ontology:
    class AdministrativeCase(sulo.Process):
        is_a = [
             sulo.hasDirectPart.max(1,Admission),
             sulo.hasDirectPart.max(1,Discharge),
             sulo.hasDirectPart.some(ClinicalVisit),
             sulo.hasDirectPart.some(MeasurementProcess),
             sulo.hasDirectPart.some(DiagnosticProcess),
             sulo.hasDirectPart.some(MedicationAdministration),
             sulo.hasParticipant.some(SubjectOfCareRole & sulo.isFeatureOf.some(HumanOrganism))
        ]
    print("Axioms added to AdministrativeCase .")

Finally, let's print the class tree:

In [ ]:

health_class_tree = get_color_tree([sulo, health_ontology])
print("Healthcare Class hierarchy")
display(health_class_tree)  # Class hierarchy

Patient Journey

John Doe ([patient_john_doe]) was admitted to the cardiology department ([cardiology_dept]) through the admission event [admission_20260320] (rdf:type: Admission) at 08:30 on March 20, 2026.
This event is linked via (atTime) to [admission_time], and involves the participants [john_patient_role] and [smith_provider_role] through (hasParticipant).
The provider role [smith_provider_role] (isFeatureOf) [dr_smith], who is responsible for the patient’s care.

Blood Pressure Measurement

Shortly after admission, a blood pressure measurement process [bp_measurement_process] (rdf:type: MeasurementProcess) took place at 09:15, linked via (atTime) to [measurement_time] and located in [cardiology_dept] (isIn).
Participants via (hasParticipant) include:

Instrument role [bp_monitor_role] (isFeatureOf → [bp_monitor])
Patient and provider roles ([john_patient_role], [smith_provider_role])
Anatomical sites [upper_arm_site] (hasPart → [left_arm]) and [left_arm]
Output roles [systolic_output_role] and [diastolic_output_role]

Measurements:

[systolic_output_role] (isFeatureOf) [systolic_measurement]
(hasValue) 145
(hasPart) [mmHg_unit]
(refersTo) [systolic_bp_quality]
[diastolic_output_role] (isFeatureOf) [diastolic_measurement]
(hasValue) 92
(hasPart) [mmHg_unit]
(refersTo) [diastolic_bp_quality]
Diagnosis

At 10:00, a diagnostic process [diagnostic_process] (rdf:type: DiagnosticProcess) occurred, linked via (atTime) to [diagnosis_time] and located in [cardiology_dept] (isIn).
Participants:

[john_patient_role], [smith_provider_role],
Output role [diagnosis_output_role] (isFeatureOf) [hypertension_diagnosis_statement]
(refersTo) [hypertension_condition]
Medication Administration

At 10:30, [medication_administration] (rdf:type: MedicationAdministration) occurred, linked via (atTime) to [medication_time].
Participants via (hasParticipant):

[john_patient_role], [smith_provider_role], [completed_status]
Pharmaceutical product [lisinopril_product]
(hasPart): [lisinopril_dose], [lisinopril_substance], [tablet_form]
Dose [lisinopril_dose] (hasValue) 10
Administration site [upper_arm_site]

Prescription [medication_prescription] (refersTo) both [lisinopril_product] and [medication_administration].

Discharge

At 14:00, the discharge event [discharge_20260320] (rdf:type: Discharge) occurred:

(atTime) → [discharge_time]
Participants (hasParticipant) → [john_patient_role], [smith_provider_role]
Location (isIn) → [cardiology_dept]
Administrative Case

All events are aggregated under [case_20260320_john_doe] (rdf:type: AdministrativeCase):

(hasDirectPart):
[admission_20260320], [bp_measurement_process], [diagnostic_process], [medication_administration], [discharge_20260320]
(hasParticipant) → [john_patient_role]

In [ ]:

with health_ontology:

    # =========================
    # --- Persons ---
    # =========================
    patient = Person("patient_john_doe")
    provider = Person("dr_smith")

    # =========================
    # --- Roles ---
    # =========================
    john_patient_role = SubjectOfCareRole("john_patient_role")
    smith_provider_role = CareProviderRole("smith_provider_role")

    john_patient_role.isFeatureOf = [patient]
    smith_provider_role.isFeatureOf = [provider]

    # =========================
    # --- Care Unit ---
    # =========================
    cardiology_dept = CareUnit("cardiology_dept")

    # =========================
    # --- Admission ---
    # =========================
    admission_time = sulo.TimeInstant("admission_time")
    admission_time.hasValue = datetime.datetime(2026, 3, 20, 8, 30)

    admission = Admission("admission_20260320")
    admission.atTime = [admission_time]
    admission.hasParticipant = [john_patient_role, smith_provider_role]
    admission.isIn = [cardiology_dept]

    # =========================
    # --- Measurement ---
    # =========================
    measurement_time = sulo.TimeInstant("measurement_time")
    measurement_time.hasValue = datetime.datetime(2026, 3, 20, 9, 15)

    # Device
    bp_monitor = Device("bp_monitor")

    bp_monitor_role = InstrumentRole("bp_monitor_role")
    bp_monitor_role.isFeatureOf = [bp_monitor]

    # Body sites
    left_arm = BodySite("left_arm")
    upper_arm_site = BodySite("upper_arm_site")
    upper_arm_site.hasPart = [left_arm]

    # Units
    mmhg = sulo.Unit("mmHg_unit")
    mmhg.hasValue = "mmHg"

    # Qualities
    systolic_quality = SystolicBloodPressure("systolic_bp_quality")
    diastolic_quality = DiastolicBloodPressure("diastolic_bp_quality")

    # Measurements
    systolic_measurement = Measurement("systolic_measurement")
    systolic_measurement.hasValue = 145
    systolic_measurement.hasPart = [mmhg]
    systolic_measurement.refersTo = [systolic_quality]
    systolic_measurement.atTime = [measurement_time]

    diastolic_measurement = Measurement("diastolic_measurement")
    diastolic_measurement.hasValue = 92
    diastolic_measurement.hasPart = [mmhg]
    diastolic_measurement.refersTo = [diastolic_quality]
    diastolic_measurement.atTime = [measurement_time]

    # Output roles
    systolic_output_role = OutputRole("systolic_output_role")
    diastolic_output_role = OutputRole("diastolic_output_role")

    systolic_output_role.isFeatureOf = [systolic_measurement]
    diastolic_output_role.isFeatureOf = [diastolic_measurement]

    # Measurement process
    bp_process = MeasurementProcess("bp_measurement_process")
    bp_process.atTime = [measurement_time]
    bp_process.hasParticipant = [
        bp_monitor_role,
        systolic_output_role,
        diastolic_output_role,
        john_patient_role,
        smith_provider_role,
        left_arm,
        upper_arm_site
    ]
    bp_process.isIn = [cardiology_dept]

    # =========================
    # --- Diagnosis ---
    # =========================
    diagnosis_time = sulo.TimeInstant("diagnosis_time")
    diagnosis_time.hasValue = datetime.datetime(2026, 3, 20, 10, 0)

    condition = Condition("hypertension_condition")

    diagnosis_statement = DiagnosisStatement("hypertension_diagnosis_statement")
    diagnosis_statement.refersTo = [condition]

    diagnosis_output_role = OutputRole("diagnosis_output_role")
    diagnosis_output_role.isFeatureOf = [diagnosis_statement]

    diagnostic_process = DiagnosticProcess("diagnostic_process")
    diagnostic_process.atTime = [diagnosis_time]
    diagnostic_process.hasParticipant = [
        diagnosis_output_role,
        john_patient_role,
        smith_provider_role
    ]
    diagnostic_process.isIn = [cardiology_dept]

    # =========================
    # --- Medication ---
    # =========================
    medication_time = sulo.TimeInstant("medication_time")
    medication_time.hasValue = datetime.datetime(2026, 3, 20, 10, 30)

    # Substance + product
    lisinopril_substance = Substance("lisinopril_substance")

    tablet_form = PharmaceuticalDoseForm("tablet_form")

    lisinopril_dose = PharmaceuticalDose("lisinopril_dose")
    lisinopril_dose.hasValue = 10

    lisinopril_product = PharmaceuticalProduct("lisinopril_product")
    lisinopril_product.hasPart = [ lisinopril_substance ]
    lisinopril_product.hasFeature = [
        lisinopril_dose,
        tablet_form
    ]

    # Status
    completed_status = Completed("completed_status")

    # Medication administration
    medication_admin = MedicationAdministration("medication_administration")
    medication_admin.atTime = [medication_time]
    medication_admin.hasParticipant = [
        completed_status,
        john_patient_role,
        smith_provider_role,
        lisinopril_product,
        upper_arm_site
    ]
    medication_admin.isIn = [cardiology_dept]

    # Prescription
    prescription = MedicationPrescription("medication_prescription")
    prescription.refersTo = [lisinopril_product, medication_admin]

    # =========================
    # --- Discharge ---
    # =========================
    discharge_time = sulo.TimeInstant("discharge_time")
    discharge_time.hasValue = datetime.datetime(2026, 3, 20, 14, 0)

    discharge = Discharge("discharge_20260320")
    discharge.atTime = [discharge_time]
    discharge.hasParticipant = [john_patient_role, smith_provider_role]
    discharge.isIn = [cardiology_dept]

    # =========================
    # --- Case ---
    # =========================
    case = AdministrativeCase("case_20260320_john_doe")
    case.hasDirectPart = [
        admission,
        bp_process,
        diagnostic_process,
        medication_admin,
        discharge
    ]
    case.hasParticipant = [john_patient_role]


Finally, we save the ontology as follows:

In [ ]:
# Save the extended ontology as a Turtle file
health_ontology.save(file = "my-health-sulo.owl", format = "rdfxml")
print("Ontology saved to 'my-health-sulo.owl' in OWL/XML format.")

Issues not covered in this tutorial:

*   Realizable entities and the processes they are grounded in: similar problem of referring of non-instantiated classes as discussed in the case of medication prescription. Examples: allergies, risks, body functions
*   Factuality statements (known as verification status in FHIR). They are just parts of Diagnosis statements. Example: negated statements, suspected or differential diagnoses
.

